Professor — Week 2 OpenAI (lay health deep research)

In [ ]:
from agents import Agent, Runner, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from typing import Annotated, Dict, List, Literal, Optional, Tuple
from datetime import datetime
import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import asyncio
import time


In [ ]:
# load environment variables with overide = true 
load_dotenv(override=True)

# set the openai api key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# set the openai api key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [ ]:
# For pushover
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = f"https://api.pushover.net/1/messages.json"

if  pushover_user:
    print("Pushoveruser found")
else:
    print("Pushover user not found")

if pushover_token:
    print("Pushover token found")
else:
    print("Pushover token not found")

if pushover_url:
    print("Pushover url found")
else:
    print("Pushover url not found")




In [ ]:
# function to this pushover url 
def pushover_notification(message):
    data = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    response = requests.post(pushover_url, data=data)
    


In [ ]:
pushover_notification("Hi PROFESSOR !!")

In [ ]:
#  Intake and guardrails

class IntakeResult(BaseModel):
  status: Literal["ok", "blocked"] = Field(
      description='Use "ok" for general, non-personal education; "blocked" for symptoms, dosing, diagnosis, or emergencies.'
  )
  block_reason: Optional[str] = Field(
      default=None,
      description="Short user-facing message when status is blocked; must be null when ok.",
  )
  condition: Optional[str] = Field(default=None, description="Topic in plain language when ok.")
  audience: Optional[str] = Field(default=None, description='e.g. "general public", "newly diagnosed adults".')
  region: Optional[str] = Field(default=None, description='e.g. "UK NHS patient info", "US general".')
  out_of_scope: Optional[str] = Field(
      default=None,
      description="What you will not do (e.g. no personal treatment advice).",
  )

INTAKE_INSTRUCTIONS = """You are the intake stage for a lay health *education* tool (not a clinician).
If the user asks for personal diagnosis, symptom triage, medication dosing, stopping/changing medication, or emergency help, set status to "blocked" and give a short, kind block_reason (suggest a clinician or emergency services if relevant). Leave condition/audience/region/out_of_scope null.
If the request is appropriate, set status to "ok" and fill:
- condition: one clear educational topic (one condition or theme).
- audience: who the reader is, in plain language.
- region: which country's patient-facing context to prefer.
- out_of_scope: bullet-style text stating no personal medical advice, no dosing, no diagnosis from symptoms.
Never provide medical advice in this stage; only classify and normalize the request."""


intake_agent = Agent(
  name="IntakeAgent",
  instructions=INTAKE_INSTRUCTIONS,
  model="gpt-4o-mini",
  output_type=IntakeResult,
)

async def run_intake(raw_user_text: str) -> IntakeResult:
  result = await Runner.run(intake_agent, raw_user_text.strip())
  return result.final_output


In [ ]:
#  stage 1 test the intake agent

async def test_stage1_intake():
    good = "Type 2 diabetes overview for adults in the UK, plain language for someone newly diagnosed"
    r_ok = await run_intake(good)

    if r_ok.status == "ok" and r_ok.condition and r_ok.audience and r_ok.region:
        print("PASS: Intake agent returned ok with condition, audience, and region")
    else:
        print("FAIL: Intake agent returned ok with condition, audience, and region")

        #Unacceptable requests  

    bad = "I have chest pain and numbness in my left arm — should I stop my aspirin and what dose of nitro should I take?"
    r_blocked = await run_intake(bad)

    if r_blocked.status == "blocked":
        print("PASS: Intake agent returned blocked for unacceptable request")
    else:
        print("FAIL: Intake agent returned blocked for unacceptable request")
    
    

In [ ]:
await test_stage1_intake()

In [ ]:
MAX_PLANNER_QUERIES = 3


class PlannerSearchItem(BaseModel):
    reason: str = Field(description="Why this web search supports one section of the lay report.")
    query: str = Field(
        description="Concrete web search query; non-personal; bias toward reputable patient sources for the brief's region."
    )


class PlannerSearchPlan(BaseModel):
    searches: Annotated[
        list[PlannerSearchItem],
        Field(min_length=1, max_length=MAX_PLANNER_QUERIES),
    ]


PLANNER_INSTRUCTIONS = f"""You plan web searches for a lay health *education* report (not personal medical advice).

From the user's structured brief, output between 1 and {MAX_PLANNER_QUERIES} searches. Each search maps to ONE section of the final report:
overview, who is affected, assessment/monitoring (general), topics to discuss with a clinician (categories only), misconceptions vs reputable sources, urgent-care messaging (general public-health only).

Each query must be a concrete search string a search engine can run — non-personal (no "you should", no dosing, no diagnosis from symptoms). Prefer wording that surfaces patient-facing sources for the brief's region (e.g. site:nhs.uk, NHS, CDC, WHO).

You do not run searches yourself; only output the plan."""


planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=PlannerSearchPlan,
)


def brief_to_planner_prompt(intake: IntakeResult) -> str:
    return (
        "Structured brief from intake (status already verified as ok):\n"
        f"- condition: {intake.condition}\n"
        f"- audience: {intake.audience}\n"
        f"- region: {intake.region}\n"
        f"- out_of_scope: {intake.out_of_scope}\n"
    )


async def run_planner(intake: IntakeResult) -> PlannerSearchPlan:
    if intake.status != "ok":
        raise ValueError("Planner requires intake.status == 'ok'")
    msg = brief_to_planner_prompt(intake)
    result = await Runner.run(planner_agent, msg)
    return result.final_output


In [ ]:
# Stage 2 planner agent

async def test_stage2_planner():
    ok = await run_intake(
        "Type 2 diabetes overview for adults in the UK, plain language for someone newly diagnosed."
    )
    assert ok.status == "ok"
    plan = await run_planner(ok)

    items = plan.searches if hasattr(plan, "searches") else plan
    n = len(items)
    if n <= MAX_PLANNER_QUERIES and n >= 1:
        print(f"PASS: planner returned {n} searches (max {MAX_PLANNER_QUERIES}).")
    else:
        print(f"FAIL: expected 1..{MAX_PLANNER_QUERIES}, got {n}.")

    for i, it in enumerate(items, 1):
        q = it.query if hasattr(it, "query") else it
        print(i, q)


await test_stage2_planner()

In [ ]:
# Stage 3 — OpenAI WebSearchTool 

SEARCH_MODEL = "gpt-4o-mini"

SEARCH_INSTRUCTIONS = """You are a research assistant for lay health *education* only (not personal medical advice).
You must use web search. Produce a concise summary under ~300 words: main points only, tight phrasing.
Include URLs or citations returned by search when available.
Do not diagnose, dose, or advise the reader personally. No preamble — summary only."""

search_agent = Agent(
    name="SearchAgent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model=SEARCH_MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)


async def run_search_for_item(item: PlannerSearchItem, intake: IntakeResult) -> dict:
    """One web search + summary per planner item."""
    message = (
        f"Research brief — condition: {intake.condition}; audience: {intake.audience}; region: {intake.region}\n"
        f"Section goal: {item.reason}\n"
        f"Search query: {item.query}\n"
        f"Out of scope for the whole project: {intake.out_of_scope}\n"
        "Summarize only what is relevant to that section for a general audience."
    )
    result = await Runner.run(search_agent, message)
    return {
        "reason": item.reason,
        "query": item.query,
        "summary": str(result.final_output),
    }


async def run_all_searches(plan: PlannerSearchPlan, intake: IntakeResult) -> list[dict]:
    """Sequential searches: predictable cost ≈ one WebSearchTool invocation per planner item."""
    bundle = []
    for item in plan.searches:
        bundle.append(await run_search_for_item(item, intake))
    print(f"Web search stages completed: {len(bundle)} search runs (WebSearchTool).")
    return bundle


def evidence_bundle_to_text(bundle: list[dict]) -> str:
    """Single string for the writer stage."""
    parts = []
    for i, b in enumerate(bundle, 1):
        parts.append(
            f"### Evidence {i}: {b['reason']}\n"
            f"**Query:** {b['query']}\n\n{b['summary']}\n"
        )
    return "\n".join(parts)


In [ ]:
# Stage 3 smoke test

async def test_stage3_search():
    ok = await run_intake(
        "Type 2 diabetes overview for adults in the UK, plain language for someone newly diagnosed."
    )
    assert ok.status == "ok"
    plan = await run_planner(ok)
    bundle = await run_all_searches(plan, ok)
    assert len(bundle) == len(plan.searches)
    for b in bundle:
        assert b["summary"].strip(), f"Empty summary for query: {b['query']}"
    print("PASS: Stage 3 returned one non-empty summary per planned query.")
    text = evidence_bundle_to_text(bundle)
    print("Evidence bundle length (chars):", len(text))


# await test_stage3_search()


In [ ]:
# Stage 4 — Writer (no tools; uses web evidence bundle + brief)

WRITER_MODEL = "gpt-4o-mini"

WRITER_INSTRUCTIONS = """You write one lay-audience Markdown report for *general education only*.

Rules:
- Start with a short **Medical disclaimer**: not medical advice; not a diagnosis; consult qualified professionals; emergency care if needed.
- Use only the structured brief and evidence blocks below (they come from web search summaries). Do not invent drugs, doses, or personal instructions beyond what the evidence supports.
- Prefer facts that appear in the evidence; when citing, reuse links from the summaries when present. Still encourage readers to verify with official patient information for their region.
- Use clear headings: Overview; Who it affects; Assessment and monitoring (general); Topics to discuss with a clinician; Common misconceptions; When to seek urgent care (general public information only); Sources and further reading.
- Tone: plain language for the stated audience. No "you should take..." directives."""

writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_INSTRUCTIONS,
    tools=[],
    model=WRITER_MODEL,
)


async def run_writer(intake: IntakeResult, evidence_markdown: str) -> str:
    user_msg = (
        f"## Brief\n- condition: {intake.condition}\n- audience: {intake.audience}\n"
        f"- region: {intake.region}\n- out_of_scope: {intake.out_of_scope}\n\n"
        f"## Evidence from web search (summaries)\n{evidence_markdown}"
    )
    result = await Runner.run(writer_agent, user_msg)
    return str(result.final_output).strip()


In [ ]:
# Full pipeline

ENABLE_PUSHOVER = os.getenv("ENABLE_PUSHOVER", "true").strip().lower() in ("1", "true", "yes")


async def run_full_pipeline(raw_user_text: str) -> Tuple[str, IntakeResult]:
    """Returns (markdown_report_or_message, intake_result). If blocked, report is the block_reason."""
    intake = await run_intake(raw_user_text.strip())
    if intake.status != "ok":
        msg = intake.block_reason or "This request cannot be processed as educational-only content."
        return msg, intake

    plan = await run_planner(intake)
    bundle = await run_all_searches(plan, intake)
    evidence_md = evidence_bundle_to_text(bundle)
    report = await run_writer(intake, evidence_md)

    if ENABLE_PUSHOVER and pushover_token and pushover_user:
        subj = (intake.condition or "Report")[:80]
        try:
            pushover_notification(f"Lay health report ready: {subj}")
        except Exception as e:
            print("Pushover notify failed:", e)

    return report, intake


In [ ]:
# Gradio UI 

def _run_ui(topic: str, region: str, audience: str) -> str:
    topic = (topic or "").strip()
    region = (region or "").strip()
    audience = (audience or "").strip()
    if not topic:
        return "Please enter a topic or condition for general education."
    raw = (
        f"Educational overview request. Topic / condition: {topic}. "
        f"Preferred region for patient-information context: {region or 'not specified'}. "
        f"Target audience: {audience or 'general public'}."
    )

    async def _go():
        return await run_full_pipeline(raw)

    report, intake = asyncio.run(_go())
    if intake.status != "ok":
        return f"**Could not run research**\n\n{report}"
    return report


demo = gr.Interface(
    fn=_run_ui,
    inputs=[
        gr.Textbox(label="Topic / condition (general education only)", lines=2),
        gr.Textbox(label="Region / health system context (e.g. UK, US)", lines=1),
        gr.Textbox(label="Audience (e.g. newly diagnosed adults)", lines=1),
    ],
    outputs=gr.Markdown(label="Report (Markdown)"),
    title="Lay health education (OpenAI Agents SDK + WebSearchTool)",
    description="Educational only — not medical advice. Uses OpenAI web search (paid); see 2_openai/4_lab4.ipynb pricing notes.",
)

demo.launch(share=False)
